# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset on human mast cell extracellular vesicle proteins, following the Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset is provided by a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# View basic metadata
print('Dataset Title:', dataset.metadata.name)
print('Description:', dataset.metadata.description)
print('Published:', dataset.metadata.datePublished)
print('Keywords:', dataset.metadata.keywords)
print('Identifier:', dataset.metadata.identifier)

## 2. Data Overview
Let's list the available record sets and their fields, referencing all by their Croissant `@id`.
We'll also display basic structure using the provided metadata methods.

In [ ]:
from pprint import pprint

# List all record sets with their @id
print('Record sets in the dataset:')
for rs in dataset.metadata.record_sets:
    print(f"  RecordSet name: {rs.name}, @id: {rs.id}")
    print("    Fields and Columns:")
    for field in rs.fields:
        col_id = field.column.id if hasattr(field, 'column') and field.column else 'N/A'
        print(f"     - Field: {field.name}, @id: {field.id}, column @id: {col_id}, type: {field.data_type}")
    print()

We will use the main record set (protein results table) for our extraction and analysis.

In [ ]:
# Define the main record set @id (replace with the actual @id printed above if needed)
# We'll select the first public record set for demonstration.
main_record_set = dataset.metadata.record_sets[0].id

print('Chosen record set @id:', main_record_set)

# Let's also grab the IDs of all numeric fields for later EDA
numeric_fields = []
str_fields = []
for f in dataset.metadata.record_sets[0].fields:
    # According to Croissant, numeric types might include 'Number', 'Integer', 'Float'
    if f.data_type.lower() in ['number', 'float', 'integer']:
        numeric_fields.append(f.id)
    else:
        str_fields.append(f.id)
print('Numeric fields @id:', numeric_fields if numeric_fields else 'None identified')
print('String/categorical fields @id:', str_fields[:5])

## 3. Data Extraction
Let's load data from the protein record set into a DataFrame for further analysis. We'll use the record set's `@id`.

In [ ]:
# Extract all records for all record sets as DataFrames indexed by @id
dataframes = {}
for rs in dataset.metadata.record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f'Loaded {len(df)} records from RecordSet: {rs.id} -- columns: {list(df.columns)}')

# Select the main record set for display
main_df = dataframes[main_record_set]
print('\nColumns in main DataFrame:')
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
We can now perform basic filtering, normalization, and grouping. We'll select a numeric field for exploration by its `@id`.

In [ ]:
# If no numeric fields were found, print warning and exit; otherwise select the first for demo.
if not numeric_fields:
    print('No numeric fields identified for EDA.')
else:
    # Use the first numeric field
    numeric_field = numeric_fields[0]
    print('Numeric field for analysis:', numeric_field)

    # Display some statistics
    vals = pd.to_numeric(main_df[numeric_field], errors='coerce')
    print(f"{numeric_field} summary stats:")
    print(vals.describe())

    # Simple threshold filtering, e.g. greater than mean
    threshold = vals.mean()
    filtered_df = main_df[vals > threshold].copy()
    print(f"\nFiltered records where {numeric_field} > {threshold:.2f} (mean):\n", filtered_df.head(3))

    # Normalization (mean 0, std 1)
    filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - vals.mean()) / vals.std()
    print(f"\nNormalized values for {numeric_field} (first 3):\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

    # Try grouping by a common attribute field (e.g., description or other string field if available)
    if str_fields:
        group_field = str_fields[0]
        print(f"\nGrouping filtered data by: {group_field}\n")
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().head(3)
        print(grouped)
    else:
        print('No string/categorical field available for grouping.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field and compare groups if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(main_df[numeric_field], errors='coerce').dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Optionally, boxplot by group
    if str_fields:
        plt.figure(figsize=(8,4))
        top_groups = main_df[str_fields[0]].value_counts().index[:4]
        grp_df = main_df[main_df[str_fields[0]].isin(top_groups)]
        sns.boxplot(x=str_fields[0], y=numeric_field, data=grp_df)
        plt.title(f'Boxplot of {numeric_field} by {str_fields[0]} (top 4 groups)')
        plt.show()

## 6. Conclusion
In this notebook we have:
* Loaded a FAIR²-compliant Croissant dataset of protein abundances from stimulated human mast cell extracellular vesicles,
* Explored the available record sets and fields using their Croissant `@id`s for unambiguous reference,
* Extracted the main protein records table as a DataFrame,
* Performed filtering, normalization, and grouping by attribute fields,
* Visualized the distribution of a key quantitative field.

To go further, consider integrating the protein accessions with UniProt for annotation, comparing abundance changes across experiment groups, or using other entity `@id`s for richer analyses.